In [5]:
from typing import List, Tuple
import pickle
import numpy as np
from scipy.linalg import eigh
import pyscf
import tenpy as tp
from tenpy.networks.mps import MPS
from tenpy.models.molecular import MolecularModel

In [8]:
def partially_filled_state(n_sites: int, n_electrons: int) -> List[str]:
    """Make a state where the first n_electrons/2 orbitals are full, and the rest are empty.
    If n_electrons is odd, then the next unoccupied site will be filled with a spin-down
    electron.
    E.g. For n_sites = 4, n_electrons = 3, returns ['full', 'down', 'empty', 'empty']."""

    assert n_electrons < 2 * n_sites
    n_full = n_electrons // 2
    if n_electrons % 2 != 0:
        # Odd number of electrons - Extra spin down in the next site.
        n_half_full = 1
    else:
        n_half_full = 0
    n_empty = n_sites - n_full - n_half_full
    product_state = ["full"] * n_full + ["down"] * n_half_full + ["empty"] * n_empty
    assert len(product_state) == n_sites
    return product_state


def get_ground_state(model: MolecularModel, n_electrons: int, max_bond: int) -> Tuple[MPS, float]:
    """Get the DMRG ground state of the molecular model."""

    # product_state = ["up", "down"] * (len(model.lat.mps_sites()) // 2) # start in semi-Néel state 
    n_sites = len(model.lat.mps_sites())
    product_state = partially_filled_state(n_sites, n_electrons)
    psi = tp.MPS.from_product_state(model.lat.mps_sites(), product_state)
    dmrg_params = {'mixer': True, 'trunc_params': {'chi_max': max_bond, 'svd_min': 1e-9},
        'max_E_err': 1e-9, 'max_S_err': 1e-6, 'min_sweeps': 20, 'max_sweeps': 50, 'max_trunc_err': None,
        'max_N_sites_per_ring': None}
    engine = tp.TwoSiteDMRGEngine(psi, model, dmrg_params)
    print(engine.options)
    e_ground, psi_ground = engine.run()
    return psi_ground, e_ground


In [3]:
# Get V_ijkl and h_ij for the HF molecule.
mol = pyscf.M(
    atom = 'H 0 0 0; F 0 0 1.1',  # in Angstrom
    basis = 'ccpvdz',
    symmetry = True,
)
orb = mol.RHF().run().mo_coeff
v_ijkl = mol.intor('int2e', aosym='s1')
h_ij = mol.intor('int1e_nuc') + mol.intor('int1e_kin')
print("One- and two-body tensor dimensions:")
print(h_ij.shape)
print(v_ijkl.shape)
# Make a TeNPy molecular model
params = {"one_body_tensor": h_ij, "two_body_tensor": v_ijkl}
mol_model = MolecularModel(params)
print(mol_model.lat.N_sites_per_ring)

converged SCF energy = -99.9873974403489
One- and two-body tensor dimensions:
(19, 19)
(19, 19, 19, 19)
19


In [9]:
max_bond = 10
n_electrons = 10 # PubChem says the formal charge of HF is 0.
ground_state, energy = get_ground_state(mol_model, n_electrons, max_bond)
print("DMRG energy =", energy)

/Users/benjamindalfavero/.venv/tdvp/lib/python3.12/site-packages/tenpy/algorithms/algorithm.py:99: TenpyInconsistencyWarning: Maximum number of sites per ring (``max_N_sites_per_ring``) exceeded.
  consistency_check(N_sites_per_ring, self.options, 'max_N_sites_per_ring', 18,


Config, name='TwoSiteDMRGEngine', options:
{'N_sweeps_check': 1,
 'chi_list': None,
 'combine': False,
 'diag_method': 'default',
 'lanczos_params': Config(<0 options>, 'lanczos_params'),
 'max_E_err': 1e-09,
 'max_N_sites_per_ring': None,
 'max_S_err': 1e-06,
 'max_sweeps': 50,
 'max_trunc_err': None,
 'min_sweeps': 20,
 'mixer': True,
 'mixer_params': Config(<3 options>, 'mixer_params'),
 'trunc_params': Config(<2 options>, 'trunc_params')}


final DMRG state not in canonical form up to norm_tol=1.00e-05: norm_err=2.16e-02


DMRG energy = -112.95337869291923


/Users/benjamindalfavero/.venv/tdvp/lib/python3.12/site-packages/tenpy/algorithms/mps_common.py:792: TenpyInconsistencyWarning: Maximum truncation error (``max_trunc_err``) exceeded.
  consistency_check(np.max(self.trunc_err_list), self.options, 'max_trunc_err', 1e-4,


In [10]:
import pickle
with open('hf_ground_state.pkl', 'wb') as f:
    pickle.dump(ground_state, f)

In [15]:
def evolve_state(psi: MPS, model: MolecularModel, chi: float, T: float = 3, dt: float = 0.2) -> List[MPS]:
    """Evolve the state for total time T with steps of size dt."""

    # parameters for each step of TDVP
    num_steps = int(T / dt)
    time_params = {'start_time': 0, 'dt': dt, 'N_steps': 1,
        'trunc_params': {'chi_max': chi, 'svd_min': 1.e-10, 'trunc_cut': None},
        'max_N_sites_per_ring': None}

    # evolve the inputted state psi in place
    engine = tp.TwoSiteTDVPEngine(psi, model, time_params)

    # Save a copy of the evolved state at each time step
    states = [psi.copy()] # initial state at t = 0
    for step in range(num_steps):
        print(f"Time = {dt*step}")
        engine.run()
        states.append(psi.copy())

    return states

In [16]:
def subspace_matrices(psi_dmrg: MPS, model: MolecularModel, chi: int, T: float, dt: float):
    """Compute subspace matrices."""
    
    # Evolve the initial state with maximum bond dimension chi_time
    states = evolve_state(psi_dmrg.copy(), model, chi=chi, T=T, dt=dt)
    
    # create overlap and target matrices
    N = len(states)
    S, H = [np.zeros((N, N), dtype=complex) for _ in range(2)]

    # fill off-diagonal elements with overlaps and hamiltonian expectation values
    for i in range(N):
        for j in range(i+1, N):
            S[i, j] = states[i].overlap(states[j]) # < vi | vj >
            H[i, j] = tp.MPOEnvironment(states[i], model.H_MPO, states[j]).full_contraction(0) # < vi | H | vj >
    H += H.conj().T
    S += S.conj().T

    # fill diagonal elements 
    for i in range(N):
        S[i, i] = states[i].overlap(states[i]).real
        H[i, i] = model.H_MPO.expectation_value(states[i]).real

    return H, S

In [17]:
H, S = subspace_matrices(ground_state, mol_model, 10, 0.1, 0.01)

/Users/benjamindalfavero/.venv/tdvp/lib/python3.12/site-packages/tenpy/algorithms/algorithm.py:99: TenpyInconsistencyWarning: Maximum number of sites per ring (``max_N_sites_per_ring``) exceeded.
  consistency_check(N_sites_per_ring, self.options, 'max_N_sites_per_ring', 18,


Time = 0.0
Time = 0.01
Time = 0.02
Time = 0.03
Time = 0.04
Time = 0.05
Time = 0.06
Time = 0.07
Time = 0.08
Time = 0.09


In [18]:
subspace_output = {
    "H": H, "S": S
}
with open("subspace_matrices.pkl", "wb") as f:
    pickle.dump(subspace_output, f)

In [8]:
with open("subspace_matrices.pkl", "rb") as f:
    subspace_output = pickle.load(f)
H = subspace_output["H"]
S = subspace_output["S"]

In [10]:
def krylov_energy(H: np.ndarray, S: np.ndarray) -> float:
    """Use the 'add a small value' method to get the ground state energy
    from the Krylov subspace matrices H and S."""


    N = S.shape[0] # size of krylov subspace
    energies = []
    
    # Try to keep successively more krylov states and plot energies that result
    for keep in range(1, N + 1):
        # keep only parts of target/overlap matrices
        print(f"Keeping {keep} Krylov states")
        H0, S0 = H[:keep, :keep], S[:keep, :keep] 
        
        # CHECK THAT OVERLAPS MATRIX IS NOT BADLY CONDITIONED
        # If it is good enough, solve generalized eigenvalue problem in krylov subspace
        # H | v > = E S | v >
        svals = np.linalg.svd(S0, compute_uv=False)
        cond = svals.max() / svals.min()
        print("Condition Number:", cond)
        try:
            vals, vecs = eigh(H0, S0)
        except np.linalg.LinAlgError:
            print("HAD CONVERGENCE PROBLEM")
            eps = 1e-12
            correction = eps*np.eye(S0.shape[0])
            vals, vecs = eigh(H0, S0 + correction)
        print(f"Current energy {np.min(vals)}")

        # record difference btwn krylov ansatz and the true ground state energy 
        energies.append(np.min(vals))
    return np.min(energies)

In [11]:
tdvp_energy = krylov_energy(H, S)
print(tdvp_energy)

Keeping 1 Krylov states
Condition Number: 1.0
Current energy -112.95337410354925
Keeping 2 Krylov states
Condition Number: 1282967.973259293
Current energy -112.95691472451935
Keeping 3 Krylov states
Condition Number: 3283278795.9282413
Current energy -112.95718077074359
Keeping 4 Krylov states
Condition Number: 548847927906.79987
Current energy -112.9572061810827
Keeping 5 Krylov states
Condition Number: 37157713531653.195
Current energy -112.95733631848773
Keeping 6 Krylov states
Condition Number: 3745559857437002.0
Current energy -112.9573649259821
Keeping 7 Krylov states
Condition Number: 1.019177601508125e+17
Current energy -112.95740539075446
Keeping 8 Krylov states
Condition Number: 8.075901118142962e+16
Current energy -230.3019116624616
Keeping 9 Krylov states
Condition Number: 8.448476647120534e+16
HAD CONVERGENCE PROBLEM
Current energy -112.9573270233566
Keeping 10 Krylov states
Condition Number: 1.0167874526097067e+17
HAD CONVERGENCE PROBLEM
Current energy -112.9573411558539